In [ ]:
#Instalación de dependencias
import time
import torch

import pandas as pd

from datasets import load_dataset
from getpass import getpass
from huggingface_hub import login
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import pipeline

In [ ]:
#Configuración de GPU
device = 0 if torch.cuda.is_available() else -1

if device == 0:
    print("GPU disponible:", torch.cuda.get_device_name(0))
else:
    print("GPU no disponible. Se utilizará CPU.")

GPU no disponible. Se utilizará CPU.


In [ ]:
#Autenticación en Hugging Face
token = getpass("Ingresa tu token de Hugging Face: ")
login(token = token)

print("Autenticación completada.\n")

Ingresa tu token de Hugging Face: ··········
Autenticación completada.



In [ ]:
#Carga de dataset
dataset = load_dataset(
    "nyu-mll/glue",
    "sst2",
    split = "validation"
)

dataset = dataset.select(range(200))

print(dataset)
print(dataset[0])

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

sst2/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Dataset({
    features: ['sentence', 'label', 'idx'],
    num_rows: 200
})
{'sentence': "it 's a charming and often affecting journey . ", 'label': 1, 'idx': 0}


In [ ]:
#Carga de modelo
modelo = "takedarn/bert-tiny-sst2"

clasificador = pipeline(
    "text-classification",
    model = modelo,
    framework = "pt",
    device = device
)

print("\nModelo cargado correctamente.\n")

config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 17.5MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]


Modelo cargado correctamente.



In [ ]:
#Inferencia y medición de latencia
textos = list(dataset["sentence"])
etiquetas_reales = list(dataset["label"])

inicio = time.time()

resultados = clasificador(
    textos,
    batch_size = 8
)

fin = time.time()

tiempo_total = fin - inicio
latencia = tiempo_total / len(textos)

print("Tiempo total:", round(tiempo_total, 2), "segundos")
print("Latencia promedio:", round(latencia, 4), "segundos por texto\n")

Tiempo total: 0.49 segundos
Latencia promedio: 0.0024 segundos por texto



In [ ]:
#Conversión de predicciones
predicciones = []

for resultado in resultados:
    if resultado["label"] == "LABEL_1":
        predicciones.append(1)
    else:
        predicciones.append(0)

In [ ]:
#Cálculo de métricas
accuracy = accuracy_score(etiquetas_reales, predicciones)

precision = precision_score(
    etiquetas_reales,
    predicciones
)

recall = recall_score(
    etiquetas_reales,
    predicciones
)

f1 = f1_score(
    etiquetas_reales,
    predicciones
)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4), "\n")

Accuracy: 0.775
Precision: 0.7547
Recall: 0.8081
F1-score: 0.7805 



In [ ]:
#Tabla resumen
resumen = pd.DataFrame({
    "Modelo": ["BERT Tiny"],
    "Registros": [len(dataset)],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1-score": [f1],
    "Latencia_seg": [latencia]
})

print(resumen)

resumen.to_csv(
    "resultados_bert_tiny.csv",
    index = False
)

      Modelo  Registros  Accuracy  Precision    Recall  F1-score  Latencia_seg
0  BERT Tiny        200     0.775   0.754717  0.808081  0.780488      0.002431
